<a href="https://colab.research.google.com/github/EconShark/23_big_data_analysis_mid/blob/main/Case_Study_1/CS01_SaleAnalytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phân tích bán hàng với PySpark

## Sales Analytics with PySpark – UCI Online Retail II

**Mục tiêu:** xử lý hơn 1 triệu giao dịch và chuyển dữ liệu thành KPI, insight và đề xuất kinh doanh.

## Dataset
- **UCI Online Retail II**
- 1,067,371 giao dịch trong hai năm
- Nguồn: https://archive.ics.uci.edu/dataset/502/online+retail+ii
- DOI: 10.24432/C5CG6D
- License: CC BY 4.0

> Dataset gốc là Excel gồm hai sheet. Phần setup dưới đây tải và chuyển từng sheet sang CSV trước khi Spark đọc.

## Cài đặt và khởi tạo Spark

In [1]:
#!pip -q install pyspark pandas openpyxl

#!pip install -U kaleido
#import plotly.io as pio
# Set the global default to SVG (or 'png')
#pio.renderers.default = "svg"

In [2]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
import plotly.express as px

spark = (SparkSession.builder
         .appName('SalesAnalyticsOnlineRetailII')
         .getOrCreate())

print('Spark version:', spark.version)

Spark version: 4.0.4


## Tải dữ liệu từ UCI và chuyển Excel sang CSV
Chạy cell này trên Colab/Jupyter có Internet. Nếu đã có `online_retail_II.csv`, có thể bỏ qua.

In [3]:
import os, zipfile, urllib.request, pandas as pd

URL = 'https://archive.ics.uci.edu/static/public/502/online%2Bretail%2Bii.zip'
ZIP_PATH = 'online_retail_II.zip'
EXTRACT_DIR = 'online_retail_II_raw'
CSV_PATH = 'online_retail_II.csv'

if not os.path.exists(CSV_PATH):
    if not os.path.exists(ZIP_PATH):
        urllib.request.urlretrieve(URL, ZIP_PATH)
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

    xlsx_files = [os.path.join(EXTRACT_DIR, f) for f in os.listdir(EXTRACT_DIR) if f.lower().endswith('.xlsx')]
    assert xlsx_files, 'Không tìm thấy file .xlsx'
    xlsx_path = xlsx_files[0]
    book = pd.ExcelFile(xlsx_path)
    first = True
    for sheet in book.sheet_names:
        print('Converting:', sheet)
        pdf = pd.read_excel(xlsx_path, sheet_name=sheet)
        pdf.to_csv(CSV_PATH, mode='w' if first else 'a', index=False, header=first)
        first = False
    print('Created:', CSV_PATH)
else:
    print('Found:', CSV_PATH)

Converting: Year 2009-2010
Converting: Year 2010-2011
Created: online_retail_II.csv


# Phần A: Data Understanding & Data Cleaning

## Bài 1. Khám phá dữ liệu
1. Đọc dữ liệu bằng spark.read.csv(...).
2. Hiển thị 20 dòng đầu, schema, số dòng và số cột.
3. Đếm số giá trị thiếu của từng cột.
4. Đếm số hóa đơn, khách hàng, sản phẩm và quốc gia phân biệt.
5. Xác định khoảng thời gian dữ liệu.

Câu hỏi: Vì sao số dòng giao dịch khác số hóa đơn? Một hóa đơn có thể chứa nhiều dòng sản phẩm như
thế nào?

In [4]:
#1. Đọc dữ liệu bằng spark.read.csv(...).
sales_raw = spark.read.option('header', True).option('inferSchema', True).csv('online_retail_II.csv')

rename_map = {
    'Invoice':'InvoiceNo',
    'Price':'UnitPrice',
    'Customer ID':'CustomerID'
}

In [5]:
df = sales_raw
for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)
print(df.columns)

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [6]:
#2. Hiển thị 20 dòng đầu, schema, số dòng và số cột.
df.show(20, truncate=False)
df.printSchema()
print('Rows:', df.count())
print('Columns:', len(df.columns))

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|489434   |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95     |13085.0   |United Kingdom|
|489434   |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75     |13085.0   |United Kingdom|
|489434   |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75     |13085.0   |United Kingdom|
|489434   |22041    |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1      |13085.0   |United Kingdom|
|489434   |21232    |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25     |13085.0   |United Kingdom|
|489434   |22064

In [7]:
#3. Đếm số giá trị thiếu của từng cột.

missing = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns])
missing.show(truncate=False)

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|0        |0        |4382       |0       |0          |0        |243007    |0      |
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [8]:
#4. Đếm số hóa đơn, khách hàng, sản phẩm và quốc gia phân biệt.

df.select(
    F.countDistinct('InvoiceNo').alias('Distinct_Invoices'),
    F.countDistinct('StockCode').alias('Distinct_Products'),
    F.countDistinct('CustomerID').alias('Distinct_Customers'),
    F.countDistinct('Country').alias('Distinct_Countries'),
    F.min('InvoiceDate').alias('Min_Date'),
    F.max('InvoiceDate').alias('Max_Date')
).show(truncate=False)

#5. Xác định khoảng thời gian dữ liệu.
df.select(
    F.min('InvoiceDate').alias('Min_Date'),
    F.max('InvoiceDate').alias('Max_Date')
).show(truncate=False)

+-----------------+-----------------+------------------+------------------+-------------------+-------------------+
|Distinct_Invoices|Distinct_Products|Distinct_Customers|Distinct_Countries|Min_Date           |Max_Date           |
+-----------------+-----------------+------------------+------------------+-------------------+-------------------+
|53628            |5305             |5942              |43                |2009-12-01 07:45:00|2011-12-09 12:50:00|
+-----------------+-----------------+------------------+------------------+-------------------+-------------------+

+-------------------+-------------------+
|Min_Date           |Max_Date           |
+-------------------+-------------------+
|2009-12-01 07:45:00|2011-12-09 12:50:00|
+-------------------+-------------------+



Câu hỏi: Vì sao số dòng giao dịch khác số hóa đơn? Một hóa đơn có thể chứa nhiều dòng sản phẩm như thế nào?

Một khách hàng trong một lần mua (1 InvoiceNo) thường mua nhiều mặt hàng khác nhau (StockCode), mỗi mặt hàng tạo thành 1 dòng.



In [9]:
# Thống kê số dòng sản phẩm trên mỗi hóa đơn
items_per_invoice = (df.groupBy('InvoiceNo')
    .agg(
        F.count('*').alias('Num_Rows'),
        F.countDistinct('StockCode').alias('Distinct_Products'),
        F.sum('Quantity').alias('Total_Quantity')
    ))

# Xem 5 hóa đơn có nhiều dòng sản phẩm nhất
items_per_invoice.orderBy(F.desc('Num_Rows')).show(5)

# Tính số dòng trung bình trên 1 hóa đơn
items_per_invoice.select(
    F.round(F.avg('Num_Rows'), 2).alias('Avg_Rows_Per_Invoice'),
    F.max('Num_Rows').alias('Max_Rows_In_One_Invoice')
).show()

+---------+--------+-----------------+--------------+
|InvoiceNo|Num_Rows|Distinct_Products|Total_Quantity|
+---------+--------+-----------------+--------------+
|   537434|    1350|              674|          3738|
|   538071|    1304|              648|          3286|
|   537638|    1202|              601|          2896|
|   537237|    1194|              595|          3216|
|   536876|    1186|              586|          3282|
+---------+--------+-----------------+--------------+
only showing top 5 rows
+--------------------+-----------------------+
|Avg_Rows_Per_Invoice|Max_Rows_In_One_Invoice|
+--------------------+-----------------------+
|                19.9|                   1350|
+--------------------+-----------------------+



## Bài 2. Làm sạch dữ liệu
6. Chuyển `InvoiceDate` sang timestamp.
7. Tạo cờ IsCancelled nếu `InvoiceNo` bắt đầu bằng C.
8. Thống kê số dòng giao dịch hủy và tỷ lệ hủy.
9. Kiểm tra `Quantity` <= 0 và `UnitPrice` <= 0.
10. Tạo DataFrame `sales_clean` chỉ giữ giao dịch bán hợp lệ: không hủy, `Quantity` > 0, `UnitPrice` > 0.
11. Giữ các dòng thiếu `CustomerID` cho phân tích tổng doanh thu, nhưng tạo thêm `sales_customer` cho các bài toán khách hàng.

In [10]:
#6. Chuyển `InvoiceDate` sang timestamp.
print("\n--- 6. Chuyển InvoiceDate sang timestamp ---")
df = df.withColumn('InvoiceDate', F.to_timestamp('InvoiceDate'))
df.show(5)

#7. Tạo column IsCancelled nếu `InvoiceNo` bắt đầu bằng C.
print("\n--- 7. Tạo cờ IsCancelled ---")
df = df.withColumn('IsCancelled', F.col('InvoiceNo').cast('string').startswith('C'))
df.show(5)

#8. Thống kê số dòng giao dịch hủy và tỷ lệ hủy.
print("\n--- 8. Thống kê giao dịch hủy ---")
total_rows = df.count()
cancelled_rows = df.filter(F.col('IsCancelled')).count()
print(f"Cancelled rows: {cancelled_rows:,} ({cancelled_rows/total_rows*100:.2f}%)")

#9. Kiểm tra `Quantity` <= 0 và `UnitPrice` <= 0.
print("\n--- 9. Kiểm tra Quantity <= 0 và UnitPrice <= 0 ---")
df.select(
    F.sum((F.col('Quantity') <= 0).cast('int')).alias('Quantity_le_0'),
    F.sum((F.col('UnitPrice') <= 0).cast('int')).alias('UnitPrice_le_0'),
    F.sum(F.col('InvoiceNo').cast('string').startswith('C').cast('int')).alias('CancelledRows')
).show(5)

#10. Tạo DataFrame `sales_clean` chỉ giữ giao dịch bán hợp lệ: không hủy, `Quantity` > 0, `UnitPrice` > 0.
print("\n--- 10. DataFrame sales_clean (Hợp lệ) ---")
sales_clean = df.filter(
    (~F.col('IsCancelled')) &
    (F.col('Quantity') > 0) &
    (F.col('UnitPrice') > 0)
)
sales_clean.show(5)

#11. Giữ các dòng thiếu `CustomerID` cho phân tích tổng doanh thu, nhưng tạo thêm `sales_customer` cho các bài toán khách hàng.
print("\n--- 11. Tạo thêm `sales_customer` cho các bài toán khách hàng ---")
sales_customer = sales_clean.filter(F.col('CustomerID').isNotNull())

print('Raw rows:', df.count())
print('Clean rows:', sales_clean.count())
print('Customer rows:', sales_customer.count())



--- 6. Chuyển InvoiceDate sang timestamp ---
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00|     6.95|   13085.0|United Kingdom|
|   489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|   13085.0|United Kingdom|
|   489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|   13085.0|United Kingdom|
|   489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|      2.1|   13085.0|United Kingdom|
|   489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00|     1.25|   13085.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+---------

#Phần B: Xây dựng KPI bán hàng


## Bài 3. Feature Engineering
12. Tạo Revenue = Quantity × UnitPrice.
13. Tạo Year, Month, YearMonth, DayOfWeek, Hour từ InvoiceDate.
14. Tạo trường Date để phân tích doanh thu theo ngày.

In [11]:
sales = (sales_clean
    .withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
    .withColumn('Year', F.year('InvoiceDate'))
    .withColumn('Month', F.month('InvoiceDate'))
    .withColumn('YearMonth', F.date_format('InvoiceDate', 'yyyy-MM'))
    .withColumn('DayOfWeek', F.date_format('InvoiceDate', 'E'))
    .withColumn('Hour', F.hour('InvoiceDate'))
    .withColumn('Date', F.to_date('InvoiceDate'))
)

sales.select('InvoiceNo','InvoiceDate','Quantity','UnitPrice','Revenue','YearMonth').show(10)

+---------+-------------------+--------+---------+------------------+---------+
|InvoiceNo|        InvoiceDate|Quantity|UnitPrice|           Revenue|YearMonth|
+---------+-------------------+--------+---------+------------------+---------+
|   489434|2009-12-01 07:45:00|      12|     6.95|              83.4|  2009-12|
|   489434|2009-12-01 07:45:00|      12|     6.75|              81.0|  2009-12|
|   489434|2009-12-01 07:45:00|      12|     6.75|              81.0|  2009-12|
|   489434|2009-12-01 07:45:00|      48|      2.1|100.80000000000001|  2009-12|
|   489434|2009-12-01 07:45:00|      24|     1.25|              30.0|  2009-12|
|   489434|2009-12-01 07:45:00|      24|     1.65|39.599999999999994|  2009-12|
|   489434|2009-12-01 07:45:00|      24|     1.25|              30.0|  2009-12|
|   489434|2009-12-01 07:45:00|      10|     5.95|              59.5|  2009-12|
|   489435|2009-12-01 07:46:00|      12|     2.55|30.599999999999998|  2009-12|
|   489435|2009-12-01 07:46:00|      12|

## Bài 4. KPI tổng quan

Tính các KPI sau và trình bày thành bảng tóm tắt:

| KPI | Công thức / cách tính |
| --- | --- |
| **Revenue** | `SUM(Revenue)` |
| **Orders** | `COUNT DISTINCT InvoiceNo` |
| **Units Sold** | `SUM(Quantity)` |
| **Customers** | `COUNT DISTINCT CustomerID` |
| **AOV** | `Revenue / Orders` |
| **Revenue per Customer** | `Revenue / Customers` |
| **Avg. Items per Order** | `Units Sold / Orders` |


**Yêu cầu phân tích:**

* Viết 3–5 câu nhận xét về quy mô hoạt động của doanh nghiệp, không chỉ in con số.

In [12]:
kpi = sales.agg(
    F.sum('Revenue').alias('Revenue'),
    F.countDistinct('InvoiceNo').alias('Orders'),
    F.sum('Quantity').alias('UnitsSold'),
    F.countDistinct('CustomerID').alias('Customers')
).first()

revenue = float(kpi['Revenue'])
orders = int(kpi['Orders'])
units = int(kpi['UnitsSold'])
customers = int(kpi['Customers'])

print(f'Revenue: £{revenue:,.2f}')
print(f'Orders: {orders:,}')
print(f'Units sold: {units:,}')
print(f'Customers: {customers:,}')
print(f'AOV: £{revenue/orders:,.2f}')
print(f'Revenue/customer: £{revenue/customers:,.2f}')
print(f'Items/order: {units/orders:,.2f}')

Revenue: £20,972,594.57
Orders: 40,077
Units sold: 11,420,305
Customers: 5,878
AOV: £523.31
Revenue/customer: £3,567.98
Items/order: 284.96


- Quy mô doanh nghiệp đạt gần £21 triệu trong 2 năm với hơn 40.000 đơn hàng và 11,4 triệu sản phẩm bán ra, cho thấy lưu lượng giao dịch rất lớn.

- Số lượng sản phẩm trung bình mỗi đơn hàng rất cao (~285 items/order) và giá trị đơn hàng trung bình (AOV đạt £523.31), trong khi đơn giá trung bình mỗi sản phẩm chỉ khoảng £1.84 (Revenue/Units).

- Đặc trưng số lượng mua/đơn lớn và đơn giá nhỏ chứng tỏ mô hình kinh doanh của công ty thiên về bán buôn (B2B / Wholesalers - các cửa hàng quà tặng nhập sỉ) hơn là bán lẻ từng món cho người tiêu dùng cuối (B2C)

# Phần C: Sales Performance Analysis


## Bài 5. Phân tích theo thời gian
15. Tính doanh thu theo tháng và sắp xếp theo thời gian.
16. Tính số đơn hàng theo tháng.
17. Tính AOV theo tháng.
18. Tìm 10 ngày có doanh thu cao nhất.
19. Phân tích doanh thu theo thứ trong tuần và theo giờ trong ngày.
20. Nhận xét về mùa vụ và thời điểm bán hàng cao điểm.

In [13]:
print(
    "15. Tính doanh thu theo tháng và sắp xếp theo thời gian.\n"
    "16. Tính số đơn hàng theo tháng.\n"
    "17. Tính AOV theo tháng."
)

monthly = (sales.groupBy('YearMonth')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders'))
    .orderBy('YearMonth'))
monthly.show(30, truncate=False)

px.line(
    monthly.toPandas(),
    x='YearMonth',
    y='Revenue',
    markers=True,
    title='YearMonth Revenue',
    labels={'Revenue': 'Revenue (£)'}
).show()


15. Tính doanh thu theo tháng và sắp xếp theo thời gian.
16. Tính số đơn hàng theo tháng.
17. Tính AOV theo tháng.
+---------+------------------+------+------+------------------+
|YearMonth|Revenue           |Orders|Units |AOV               |
+---------+------------------+------+------+------------------+
|2009-12  |825685.7600000007 |1682  |426981|490.8952199762192 |
|2010-01  |652708.5020000001 |1105  |391525|590.6864271493214 |
|2010-02  |553339.7360000005 |1201  |382781|460.7325029142385 |
|2010-03  |833570.1309999995 |1681  |527401|495.87753182629353|
|2010-04  |681528.9919999992 |1462  |368198|466.1621012311896 |
|2010-05  |659858.8600000002 |1500  |397206|439.9059066666668 |
|2010-06  |752270.1400000008 |1645  |408636|457.30707598784244|
|2010-07  |650712.9399999998 |1529  |338920|425.58073250490503|
|2010-08  |697274.91         |1425  |473420|489.3157263157895 |
|2010-09  |924333.0109999999 |1839  |585357|502.628064709081  |
|2010-10  |1165483.9099999995|2301  |622095|506.51191

In [14]:
print("\n--- 18. Tìm 10 ngày có doanh thu cao nhất.--- ")

daily = (sales.groupBy('Date')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.round(F.col('Revenue')/F.col('Orders'), 0))
    .orderBy(F.desc('Revenue')))
daily.show(10, truncate=False)


--- 18. Tìm 10 ngày có doanh thu cao nhất.--- 
+----------+------------------+------+------+------+
|Date      |Revenue           |Orders|Units |AOV   |
+----------+------------------+------+------+------+
|2011-12-09|200938.6          |44    |93980 |4567.0|
|2010-12-07|199236.39999999997|84    |50440 |2372.0|
|2010-09-27|118909.67000000001|95    |125534|1252.0|
|2010-12-01|117921.58         |127   |53838 |929.0 |
|2011-11-14|114419.88999999996|107   |47353 |1069.0|
|2010-12-06|109660.91999999998|102   |43590 |1075.0|
|2011-09-20|109612.02999999997|70    |43726 |1566.0|
|2010-12-09|107172.35999999996|107   |39860 |1002.0|
|2010-11-15|105185.54         |103   |38084 |1021.0|
|2011-01-18|95978.05          |41    |82978 |2341.0|
+----------+------------------+------+------+------+
only showing top 10 rows


In [15]:
print("\n--- 19. Phân tích doanh thu theo thứ trong tuần và theo giờ trong ngày.---")

day_order = (
    F.when(F.col("DayofWeek") == "Mon", 1)
    .when(F.col("DayofWeek") == "Tue", 2)
    .when(F.col("DayofWeek") == "Wed", 3)
    .when(F.col("DayofWeek") == "Thu", 4)
    .when(F.col("DayofWeek") == "Fri", 5)
    .when(F.col("DayofWeek") == "Sat", 6)
    .when(F.col("DayofWeek") == "Sun", 7)
)

dayofweek = (sales.groupBy('DayofWeek')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.round(F.col('Revenue')/F.col('Orders'), 0))
    .orderBy(day_order.asc()))
dayofweek.show(truncate=False)


px.line(
    dayofweek.toPandas(),
    x='DayofWeek',
    y='Revenue',
    markers=True,
    title='DayofWeek Revenue',
    labels={'Revenue': 'Revenue (£)'}
).show()




--- 19. Phân tích doanh thu theo thứ trong tuần và theo giờ trong ngày.---
+---------+------------------+------+-------+-----+
|DayofWeek|Revenue           |Orders|Units  |AOV  |
+---------+------------------+------+-------+-----+
|Mon      |3644756.5859999917|6351  |2057562|574.0|
|Tue      |4185862.912000003 |7291  |2173223|574.0|
|Wed      |3601790.3730000053|7185  |2054847|501.0|
|Thu      |4306392.172000006 |8288  |2377260|520.0|
|Fri      |3376979.1829999965|6097  |1698470|554.0|
|Sat      |9803.050000000001 |30    |5119   |327.0|
|Sun      |1847010.2920000034|4835  |1053824|382.0|
+---------+------------------+------+-------+-----+



In [16]:
hour = (sales.groupBy('Hour')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.round(F.col('Revenue')/F.col('Orders'), 0))
    .orderBy(F.asc('Hour')))
hour.show(truncate=False)

px.line(
    hour.toPandas(),
    x='Hour',
    y='Revenue',
    markers=True,
    title='Hour Revenue',
    labels={'Revenue': 'Revenue (£)'}
).show()

+----+------------------+------+-------+-----+
|Hour|Revenue           |Orders|Units  |AOV  |
+----+------------------+------+-------+-----+
|6   |4.25              |1     |1      |4.0  |
|7   |76232.57          |77    |45031  |990.0|
|8   |532450.6799999994 |998   |298268 |534.0|
|9   |1802195.0109999967|2754  |930598 |654.0|
|10  |2629271.8030000073|4610  |1512937|570.0|
|11  |2584521.4529999937|4903  |1472664|527.0|
|12  |2910225.350999995 |6427  |1697150|453.0|
|13  |2602570.2540000007|5695  |1578508|457.0|
|14  |2385595.470999992 |5071  |1214060|470.0|
|15  |2478204.4819999984|4566  |1271841|543.0|
|16  |1641125.3209999993|2718  |809521 |604.0|
|17  |879212.1219999994 |1407  |371828 |625.0|
|18  |282138.99000000005|466   |124094 |605.0|
|19  |128991.00999999997|332   |71970  |389.0|
|20  |39855.799999999996|53    |21834  |752.0|
+----+------------------+------+-------+-----+



20. Nhận xét về mùa vụ và thời điểm bán hàng cao điểm.

- Theo tháng: Doanh thu có tính mùa vụ (seasonality) cực kỳ rõ rệt, tăng vọt vào Quý 4 (Tháng 9, 10, đạt đỉnh vào Tháng 11 với ~£1.47M năm 2010 và ~£1.51M năm 2011) do các đại lý/khách hàng nhập hàng chuẩn bị cho lễ Giáng Sinh (Christmas).  
- Theo ngày trong tuần & giờ: Giao dịch tập trung từ Thứ 2 đến Thứ 6 (cao điểm vào Thứ 3, Thứ 5), hầu như không có/rất ít giao dịch vào Thứ 7 (càng củng cố giả thuyết khách hàng là doanh nghiệp B2B làm việc giờ hành chính). Giờ cao điểm đặt hàng là từ 10h sáng đến 15h chiều.

## Bài 6. Phân tích sản phẩm

21. Top 20 sản phẩm theo doanh thu.
22. Top 20 sản phẩm theo số lượng bán.
23. Top 20 sản phẩm theo số đơn hàng chứa sản phẩm.
24. Tìm các sản phẩm có doanh thu cao nhưng số lượng bán không cao và giải thích.
25. Tính tỷ trọng doanh thu của Top 10, Top 20 và Top 100 sản phẩm.

Câu hỏi kinh doanh: Doanh nghiệp có phụ thuộc quá nhiều vào một nhóm nhỏ sản phẩm hay không?


In [17]:
print("\n--- 21. Top 20 sản phẩm theo doanh thu ---")
product_sales = (sales.groupBy('StockCode','Description')
    .agg(
        F.round(F.sum('Revenue'),0).alias('Revenue'),
        F.sum('Quantity').alias('Units'),
        F.countDistinct('InvoiceNo').alias('Orders')
    ))
product_sales.orderBy(F.desc('Revenue')).show(20, truncate=False)

px.bar(
    product_sales.orderBy("Revenue", ascending=False).limit(20).orderBy("Revenue").toPandas(),
    x="Revenue",
    y="Description",
    orientation="h",
    title="Top 20 Products by Revenue",
    labels={"Revenue": "Revenue (£)", "Description": "Product"},
    hover_data=["StockCode", "Units", "Orders"],
).show()


--- 21. Top 20 sản phẩm theo doanh thu ---
+---------+-----------------------------------+--------+-----+------+
|StockCode|Description                        |Revenue |Units|Orders|
+---------+-----------------------------------+--------+-----+------+
|22423    |REGENCY CAKESTAND 3 TIER           |344563.0|27577|3918  |
|M        |Manual                             |340716.0|10050|784   |
|DOT      |DOTCOM POSTAGE                     |322657.0|1436 |1415  |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER |262931.0|96086|5356  |
|23843    |PAPER CRAFT , LITTLE BIRDIE        |168470.0|80995|1     |
|47566    |PARTY BUNTING                      |149187.0|28378|2674  |
|85099B   |JUMBO BAG RED RETROSPOT            |148824.0|78860|3245  |
|84879    |ASSORTED COLOUR BIRD ORNAMENT      |132188.0|81809|2807  |
|POST     |POSTAGE                            |127597.0|5461 |1851  |
|22086    |PAPER CHAIN KIT 50'S CHRISTMAS     |123142.0|36581|2018  |
|79321    |CHILLI LIGHTS                      

In [18]:
print("\n--- 22. Top 20 sản phẩm theo số lượng bán ---")

product_sales.orderBy(F.desc('Units')).show(20, truncate=False)

px.bar(
    product_sales.orderBy("Units", ascending=False).limit(20).orderBy("Units").toPandas(),
    x="Units",
    y="Description",
    orientation="h",
    title="Top 20 Products by Quantity",
    labels={"Units": "Quantity", "Description": "Product"},
    hover_data=["StockCode", "Revenue", "Orders"],
).show()


--- 22. Top 20 sản phẩm theo số lượng bán ---
+---------+-----------------------------------+--------+------+------+
|StockCode|Description                        |Revenue |Units |Orders|
+---------+-----------------------------------+--------+------+------+
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS  |25260.0 |110138|1019  |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER |262931.0|96086 |5356  |
|84879    |ASSORTED COLOUR BIRD ORNAMENT      |132188.0|81809 |2807  |
|23843    |PAPER CRAFT , LITTLE BIRDIE        |168470.0|80995 |1     |
|85099B   |JUMBO BAG RED RETROSPOT            |148824.0|78860 |3245  |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR     |81701.0 |78033 |247   |
|17003    |BROCADE RING PURSE                 |14959.0 |71430 |456   |
|21977    |PACK OF 60 PINK PAISLEY CAKE CASES |28491.0 |56794 |1993  |
|84991    |60 TEATIME FAIRY CAKE CASES        |27404.0 |54716 |2127  |
|22197    |SMALL POPCORN HOLDER               |44090.0 |49948 |1407  |
|21212    |PACK OF 72 RETROSPO

In [19]:
print("\n--- 23. Top 20 sản phẩm theo số đơn hàng chứa sản phẩm ---")

product_sales.orderBy(F.desc('Orders')).show(20, truncate=False)

px.bar(
    product_sales.orderBy("Orders", ascending=False).limit(20).orderBy("Orders").toPandas(),
    x="Orders",
    y="Description",
    orientation="h",
    title="Top 20 Products by Quantity",
    labels={"Orders": "Quantity", "Description": "Product"},
    hover_data=["StockCode", "Revenue", "Orders"],
).show()


--- 23. Top 20 sản phẩm theo số đơn hàng chứa sản phẩm ---
+---------+----------------------------------+--------+-----+------+
|StockCode|Description                       |Revenue |Units|Orders|
+---------+----------------------------------+--------+-----+------+
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|262931.0|96086|5356  |
|22423    |REGENCY CAKESTAND 3 TIER          |344563.0|27577|3918  |
|85099B   |JUMBO BAG RED RETROSPOT           |148824.0|78860|3245  |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |132188.0|81809|2807  |
|47566    |PARTY BUNTING                     |149187.0|28378|2674  |
|20727    |LUNCH BAG  BLACK SKULL.           |46422.0 |27126|2351  |
|21931    |JUMBO STORAGE BAG SUKI            |62285.0 |29503|2329  |
|21232    |STRAWBERRY CERAMIC TRINKET BOX    |48627.0 |37610|2310  |
|22411    |JUMBO SHOPPER VINTAGE RED PAISLEY |58275.0 |27732|2192  |
|22469    |HEART OF WICKER SMALL             |51128.0 |29037|2151  |
|84991    |60 TEATIME FAIRY CAKE CASES     

In [20]:
print("\n--- 24. Tìm các sản phẩm có doanh thu cao nhưng số lượng bán không cao và giải thích ---")

# Calculate summary statistics for the 'Units' column including min, max, and percentiles (median is 50%)
product_sales.select('Units').summary().show()

# Calculate the mean of Units dynamically
mean_units = product_sales.select(F.avg('Units')).first()[0]
print(f"Mean Units: {mean_units:.2f}\n")

high_rev_low_units = (product_sales
    .withColumn('AvgPrice', F.round(F.col('Revenue') / F.col('Units'), 2))
    #.filter(F.col('Units') < mean_units)
    .orderBy(F.desc('Revenue')))

high_rev_low_units.show(15, truncate=False)



--- 24. Tìm các sản phẩm có doanh thu cao nhưng số lượng bán không cao và giải thích ---
+-------+------------------+
|summary|             Units|
+-------+------------------+
|  count|              5630|
|   mean|2028.4733570159858|
| stddev| 5204.789761400969|
|    min|                 1|
|    25%|                78|
|    50%|               478|
|    75%|              1834|
|    max|            110138|
+-------+------------------+

Mean Units: 2028.47

+---------+-----------------------------------+--------+-----+------+--------+
|StockCode|Description                        |Revenue |Units|Orders|AvgPrice|
+---------+-----------------------------------+--------+-----+------+--------+
|22423    |REGENCY CAKESTAND 3 TIER           |344563.0|27577|3918  |12.49   |
|M        |Manual                             |340716.0|10050|784   |33.9    |
|DOT      |DOTCOM POSTAGE                     |322657.0|1436 |1415  |224.69  |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER |262931.0|96086|5356

**Giải thích: Sản phẩm có doanh thu cao nhưng số lượng bán không cao**

Dựa vào bảng kết quả của Câu 24, đặc biệt là cột `AvgPrice` (doanh thu chia cho số lượng), ta có thể nhận thấy một số mã `StockCode` có doanh thu cao dù số lượng ghi nhận tương đối thấp.

- **DOT (DOTCOM POSTAGE):** Doanh thu khoảng £322.657, với 1.436 đơn vị ghi nhận. Giá trị trung bình đạt khoảng £224,69 mỗi đơn vị.
- **M (Manual):** Doanh thu khoảng £340.716, với 10.050 đơn vị ghi nhận. Giá trị trung bình đạt khoảng £33,90 mỗi đơn vị.
- **POST (POSTAGE):** Doanh thu khoảng £127.597, với 5.461 đơn vị ghi nhận. Giá trị trung bình đạt khoảng £23,37 mỗi đơn vị.

**Giải thích:** Những mã này không nhất thiết đại diện cho hàng hóa vật lý thông thường. `POST` và `DOT` liên quan đến phí vận chuyển hoặc bưu phí, trong khi `M` thường được dùng để ghi nhận các giao dịch thủ công hoặc điều chỉnh trên hệ thống.

Các khoản phí và điều chỉnh này có thể có giá trị cao hơn nhiều so với giá của những mặt hàng thông thường. Vì vậy, dù số lượng ghi nhận không lớn, chúng vẫn có thể đóng góp đáng kể vào tổng doanh thu.

**Ý nghĩa đối với phân tích:** Doanh nghiệp cần phân biệt hàng hóa thực tế với phí dịch vụ và các khoản điều chỉnh trước khi xếp hạng sản phẩm theo doanh thu. Nếu gộp tất cả các mã vào cùng một bảng xếp hạng, kết quả có thể không phản ánh chính xác những mặt hàng vật lý bán chạy nhất.

In [21]:
print("\n--- 25. Tính tỷ trọng doanh thu của Top 10, Top 20 và Top 100 sản phẩm ---")

total_rev = sales.agg(F.sum('Revenue')).first()[0]
for k in [10, 20, 100]:
    top_k_rev = product_sales.orderBy(F.desc('Revenue')).limit(k).agg(F.sum('Revenue')).first()[0]
    print(f"Top {k} products share: {top_k_rev / total_rev * 100:.2f}%")


--- 25. Tính tỷ trọng doanh thu của Top 10, Top 20 và Top 100 sản phẩm ---
Top 10 products share: 10.11%
Top 20 products share: 13.43%
Top 100 products share: 29.28%


**Câu hỏi kinh doanh: Doanh nghiệp có phụ thuộc quá nhiều vào một nhóm nhỏ sản phẩm hay không?**

Câu trả lời: Có. Kết quả phân tích từ Bài 6 cho thấy doanh thu của doanh nghiệp tập trung khá nhiều vào một nhóm tương đối nhỏ các sản phẩm chủ lực.

**Dẫn chứng từ dữ liệu**
:
- Chỉ riêng **Top 10 sản phẩm** đã chiếm **10.11%** tổng doanh thu toàn công ty.
- **Top 20 sản phẩm** chiếm **13.43%** tổng doanh thu.
- **Top 100 sản phẩm** (chiếm một phần rất nhỏ trong tổng số hàng ngàn mã sản phẩm) đóng góp tới **29.28%** tổng doanh thu.


Như vậy, mặc dù danh mục mặt hàng có thể lên tới hơn 5.000 sản phẩm, sự sống còn về mặt doanh thu của doanh nghiệp vẫn phụ thuộc rất lớn vào một nhóm nhỏ các sản phẩm nằm ở Top 100 này.

## Bài 7.  Phân tích thị trường theo quốc gia
26. Doanh thu và số đơn hàng theo Country.
27. Top 10 quốc gia ngoài United Kingdom theo Revenue.
28. AOV theo Country với điều kiện quốc gia có ít nhất 50 đơn hàng.
29. So sánh UK và thị trường quốc tế.

In [22]:
print("\n--- 26. Doanh thu và số đơn hàng theo Country ---")
country_sales = (sales.groupBy('Country')
    .agg(
        F.round(F.sum('Revenue'),0).alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.countDistinct('CustomerID').alias('Customers')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders'))
    .orderBy(F.desc('Revenue')))
country_sales.show(20, truncate=False)


--- 26. Doanh thu và số đơn hàng theo Country ---
+---------------+-----------+------+---------+------------------+
|Country        |Revenue    |Orders|Customers|AOV               |
+---------------+-----------+------+---------+------------------+
|United Kingdom |1.7870978E7|36535 |5350     |489.14679074859725|
|EIRE           |664432.0   |626   |5        |1061.3929712460065|
|Netherlands    |554232.0   |228   |22       |2430.842105263158 |
|Germany        |431262.0   |789   |107      |546.5931558935362 |
|France         |356945.0   |622   |95       |573.8665594855305 |
|Australia      |169968.0   |95    |15       |1789.1368421052632|
|Spain          |109179.0   |154   |41       |708.9545454545455 |
|Switzerland    |101011.0   |93    |22       |1086.1397849462367|
|Sweden         |91904.0    |105   |19       |875.2761904761904 |
|Denmark        |69862.0    |43    |12       |1624.6976744186047|
|Belgium        |65753.0    |149   |29       |441.2953020134228 |
|Norway         |60110.0 

In [23]:
print("\n--- 27. Top 10 quốc gia ngoài United Kingdom theo Revenue ---")
country_sales.filter(F.col('Country') != 'United Kingdom').orderBy(F.desc('Revenue')).show(10, truncate=False)

print("\n--- 28. AOV theo Country với điều kiện quốc gia có ít nhất 50 đơn hàng ---")
country_sales.filter(F.col('Orders') >= 50).orderBy(F.desc('AOV')).show(20, truncate=False)


--- 27. Top 10 quốc gia ngoài United Kingdom theo Revenue ---
+-----------+--------+------+---------+------------------+
|Country    |Revenue |Orders|Customers|AOV               |
+-----------+--------+------+---------+------------------+
|EIRE       |664432.0|626   |5        |1061.3929712460065|
|Netherlands|554232.0|228   |22       |2430.842105263158 |
|Germany    |431262.0|789   |107      |546.5931558935362 |
|France     |356945.0|622   |95       |573.8665594855305 |
|Australia  |169968.0|95    |15       |1789.1368421052632|
|Spain      |109179.0|154   |41       |708.9545454545455 |
|Switzerland|101011.0|93    |22       |1086.1397849462367|
|Sweden     |91904.0 |105   |19       |875.2761904761904 |
|Denmark    |69862.0 |43    |12       |1624.6976744186047|
|Belgium    |65753.0 |149   |29       |441.2953020134228 |
+-----------+--------+------+---------+------------------+
only showing top 10 rows

--- 28. AOV theo Country với điều kiện quốc gia có ít nhất 50 đơn hàng ---
+---------

In [24]:
#Top 20 Countries by Revenue
px.pie(country_sales.limit(20).toPandas(), names='Country', values='Revenue', title='Top 20 Countries by Revenue', hole=0.3).show()

# 27. Top 10 countries outside UK by Revenue
px.pie(country_sales.filter(F.col('Country') != 'United Kingdom').orderBy(F.desc('Revenue')).limit(10).toPandas(), names='Country', values='Revenue', title='Top 10 Countries (Excl. UK) Revenue Share', hole=0.3).show()

# 28. AOV by Country (Orders >= 50)
px.bar(country_sales.filter(F.col('Orders') >= 50).orderBy(F.desc('AOV')).limit(20).toPandas(), x='Country', y='AOV', title='AOV by Country (Min 50 Orders)').show()

29. So sánh UK và thị trường quốc tế

- UK chiếm áp đảo (~85.2% tổng doanh thu với £17.87M), nhưng AOV của UK khá thấp (£489) vì bao gồm cả khách lẻ.   
- Thị trường quốc tế tiềm năng nhất:Xét về quy mô đại lý sỉ (AOV cao, $\ge 50$ orders): Hà Lan (Netherlands) có AOV cao kỷ lục (£2,430.84 từ 22 khách hàng) và Úc (Australia) có AOV £1,789.14, EIRE (Ireland) đạt doanh thu ngoài UK cao nhất (£664,431, AOV £1,061) chỉ từ 5 khách hàng định danh (các nhà phân phối độc quyền lớn).   Xét về mở rộng tệp khách hàng (nhiều khách hàng và đơn hàng ổn định): Đức (Germany - 107 khách, 789 đơn) và Pháp (France - 95 khách, 622 đơn) là 2 thị trường láng giềng có độ phủ tốt nhất để mở rộng marketing.

# Phần D: Customer Analytics & RFM

## Bài 8. Phân tích khách hàng
30. Top 20 khách hàng theo doanh thu.
31. Số đơn hàng và doanh thu trung bình trên khách hàng.
32. Tính tỷ trọng doanh thu của Top 1%, 5% và 10% khách hàng.
33. Phân tích mức độ tập trung doanh thu theo khách hàng.

In [25]:
print("\n--- 30. Top 20 khách hàng theo doanh thu ---")
print("\n--- 31. Số đơn hàng và doanh thu trung bình trên khách hàng ---")

cust = (sales_customer
    .withColumn('Revenue', F.col('Quantity')*F.col('UnitPrice'))
    .groupBy('CustomerID')
    .agg(
        F.sum('Revenue').alias('Revenue'),
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Quantity').alias('Units')
    )
    .withColumn('AOV', F.col('Revenue')/F.col('Orders')))
cust.orderBy(F.desc('Revenue')).show(20)


--- 30. Top 20 khách hàng theo doanh thu ---

--- 31. Số đơn hàng và doanh thu trung bình trên khách hàng ---
+----------+------------------+------+------+------------------+
|CustomerID|           Revenue|Orders| Units|               AOV|
+----------+------------------+------+------+------------------+
|   18102.0| 608821.6499999999|   145|188340|4198.7699999999995|
|   14646.0| 528602.5199999999|   151|367193|3500.6789403973503|
|   14156.0|313946.36999999994|   156|165992|2012.4767307692305|
|   14911.0|295972.63000000006|   398|149987| 743.6498241206032|
|   17450.0|246973.08999999994|    51| 84720|4842.6096078431365|
|   13694.0|         196482.81|   143|189205|1374.0056643356643|
|   17511.0|175603.55000000002|    60|119656|2926.7258333333334|
|   16446.0|          168472.5|     2| 80997|          84236.25|
|   16684.0|         147142.77|    55|104810|2675.3230909090908|
|   12415.0|144458.36999999997|    28| 91447| 5159.227499999999|
|   15061.0|         137818.52|   127| 80711

In [26]:
px.bar(cust.withColumn('CustomerID', F.col('CustomerID').cast('int').cast('string')).orderBy(F.desc('Revenue')).limit(20).toPandas(), x='CustomerID', y='Revenue', title='Top 20 Customers by Revenue').show()

In [27]:
import math

print("\n--- 32. Tính tỷ trọng doanh thu của Top 1%, 5% và 10% khách hàng ---")

total_customers = cust.count()
total_rev_cust = cust.agg(F.sum('Revenue')).first()[0]
for percent in [1, 5, 10]:
    limit_k = max(1, math.ceil(total_customers * percent / 100))
    top_k_rev = (
        cust
        .orderBy(F.desc("Revenue"))
        .limit(limit_k)
        .agg(F.sum("Revenue"))
        .first()[0]
    )

    share = top_k_rev / total_rev_cust * 100

    print(
        f"Top {percent}% khách hàng ({limit_k} khách hàng) "
        f"chiếm {share:.2f}% tổng doanh thu khách hàng"
    )




--- 32. Tính tỷ trọng doanh thu của Top 1%, 5% và 10% khách hàng ---
Top 1% khách hàng (59 khách hàng) chiếm 31.94% tổng doanh thu khách hàng
Top 5% khách hàng (294 khách hàng) chiếm 51.99% tổng doanh thu khách hàng
Top 10% khách hàng (588 khách hàng) chiếm 63.93% tổng doanh thu khách hàng


In [28]:
print("\n--- 33. Phân tích mức độ tập trung doanh thu theo khách hàng ---")

total_cust = cust.count()
total_cust_rev = cust.agg(F.sum('Revenue')).first()[0]

w_cust = Window.orderBy(F.desc('Revenue'))
cust_ranked = (cust
    .withColumn('Rank', F.row_number().over(w_cust))
    .withColumn('CumRevenue', F.sum('Revenue').over(w_cust.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn('CumShare', F.col('CumRevenue') / F.lit(total_cust_rev))
    .withColumn('CustPct', F.col('Rank') / F.lit(total_cust)))

for pct in [0.01, 0.05, 0.10]:
    cutoff_rank = int(total_cust * pct)
    share = cust_ranked.filter(F.col('Rank') == cutoff_rank).select('CumShare').first()[0]
    print(f"Top {int(pct*100)}% customers ({cutoff_rank} customers) generate: {share*100:.2f}% of customer revenue")


--- 33. Phân tích mức độ tập trung doanh thu theo khách hàng ---
Top 1% customers (58 customers) generate: 31.77% of customer revenue
Top 5% customers (293 customers) generate: 51.93% of customer revenue
Top 10% customers (587 customers) generate: 63.89% of customer revenue


Chỉ riêng Top 1% khách hàng (58 khách) đã đóng góp tới hơn 30% doanh thu, Top 5% đóng góp hơn 50% và Top 10% đóng góp hơn 60%. Doanh thu cực kỳ tập trung vào nhóm khách hàng này. Nếu mất đi vài khách hàng trong nhóm Top 1% này, doanh thu công ty sẽ sụt giảm nghiêm trọng.

## Bài 9. RFM Analysis

Với từng **CustomerID**, tính các chỉ số sau:

| Chỉ số | Định nghĩa |
| --- | --- |
| **Recency** | Số ngày từ lần mua gần nhất tới ngày tham chiếu = `max(InvoiceDate) + 1 ngày` |
| **Frequency** | Số hóa đơn khác nhau của khách hàng |
| **Monetary** | Tổng Revenue của khách hàng |



**Chấm điểm $R, F, M$:** Phân thành 5 nhóm (thang điểm 1–5) bằng hàm `ntile` hoặc `percentile`:
```python
w_r = Window.orderBy(F.col("Recency").desc())
w_f = Window.orderBy(F.col("Frequency"))
w_m = Window.orderBy(F.col("Monetary"))
```
**Tạo `RFMScore` & Phân nhóm khách hàng:** Áp dụng quy tắc đơn giản để chia thành các nhóm:
  - *Champions*
  - *Loyal*
  - *Potential Loyalist*
  - *At Risk*
  - *Lost/Other*


**Yêu cầu**

Với mỗi segment, thực hiện:
- **Thống kê:**
  - Số lượng khách hàng (`Customer Count`)
  - Tổng doanh thu (`Total Revenue`)
  - Doanh thu trung bình (`Average Revenue`)
  - Tần suất mua hàng trung bình (`Average Frequency`)
- **Chiến lược:**
  - Đề xuất một hành động tiếp thị (marketing action) phù hợp cho từng nhóm.



In [29]:
max_date = sales_customer.agg(F.max('InvoiceDate')).first()[0]
reference_date = max_date

rfm = (sales_customer
    .withColumn('Revenue', F.col('Quantity')*F.col('UnitPrice'))
    .groupBy('CustomerID')
    .agg(
        F.datediff(F.lit(reference_date), F.max('InvoiceDate')).alias('Recency'),
        F.countDistinct('InvoiceNo').alias('Frequency'),
        F.sum('Revenue').alias('Monetary')
    ))
rfm.show(10)

+----------+-------+---------+------------------+
|CustomerID|Recency|Frequency|          Monetary|
+----------+-------+---------+------------------+
|   12467.0|    423|        1|132.79999999999998|
|   13094.0|     21|       13| 2433.120000000001|
|   17884.0|      3|       20|           3072.89|
|   17633.0|     31|        7|           2057.39|
|   14094.0|    439|        1|335.21999999999997|
|   15893.0|    481|        1|305.28000000000003|
|   14269.0|    439|        1|            295.73|
|   16596.0|     15|        4|            579.63|
|   14285.0|     21|        8|3284.4200000000005|
|   16916.0|     23|        4|1123.3999999999996|
+----------+-------+---------+------------------+
only showing top 10 rows


In [30]:
# Chấm điểm 1–5 bằng ntile
w_r = Window.orderBy(F.col('Recency').desc())   # recency thấp sẽ nhận score cao
w_f = Window.orderBy(F.col('Frequency'))
w_m = Window.orderBy(F.col('Monetary'))

rfm_scored = (rfm
    .withColumn('R', F.ntile(5).over(w_r))
    .withColumn('F', F.ntile(5).over(w_f))
    .withColumn('M', F.ntile(5).over(w_m))
    .withColumn('RFMScore', F.concat_ws('', 'R','F','M')))
rfm_scored.show(10)

+----------+-------+---------+-----------------+---+---+---+--------+
|CustomerID|Recency|Frequency|         Monetary|  R|  F|  M|RFMScore|
+----------+-------+---------+-----------------+---+---+---+--------+
|   14095.0|    722|        1|             2.95|  1|  1|  1|     111|
|   13788.0|    505|        1|             3.75|  1|  1|  1|     111|
|   16738.0|    297|        1|             3.75|  2|  1|  1|     211|
|   14792.0|     63|        1|              6.2|  3|  2|  1|     321|
|   15913.0|    534|        1|6.300000000000001|  1|  1|  1|     111|
|   15040.0|    541|        1|             7.49|  1|  1|  1|     111|
|   18115.0|    697|        1|              9.7|  1|  1|  1|     111|
|   17378.0|    374|        1|            10.95|  2|  1|  1|     211|
|   17956.0|    249|        1|            12.75|  2|  1|  1|     211|
|   16878.0|     84|        1|             13.3|  3|  2|  1|     321|
+----------+-------+---------+-----------------+---+---+---+--------+
only showing top 10 

Tạo cột `Segment` theo logic gợi ý:
- Champions: R≥4, F≥4, M≥4
- Loyal: F≥4 và R≥3
- Potential Loyalist: R≥4 và F=2–3
- At Risk: R≤2 và F≥3
- Lost/Other: còn lại

Sau đó tính số khách hàng, doanh thu và Monetary trung bình theo segment.

In [31]:
rfm_segments = rfm_scored.withColumn(
    'Segment',
    F.when((F.col('R')>=4)&(F.col('F')>=4)&(F.col('M')>=4), 'Champions')
     .when((F.col('F')>=4)&(F.col('R')>=3), 'Loyal')
     .when((F.col('R')>=4)&(F.col('F').between(2,3)), 'Potential Loyalist')
     .when((F.col('R')<=2)&(F.col('F')>=3), 'At Risk')
     .otherwise('Lost/Other')
)

rfm_segments.groupBy('Segment').agg(
    F.count('*').alias('Customers'),
    F.sum('Monetary').alias('Revenue'),
    F.avg('Monetary').alias('AvgMonetary'),
    F.avg('Frequency').alias('AvgFrequency'),
    F.avg('Recency').alias('AvgRecency')
).orderBy(F.desc('Revenue')).show(truncate=False)

+------------------+---------+--------------------+------------------+------------------+-----------------+
|Segment           |Customers|Revenue             |AvgMonetary       |AvgFrequency      |AvgRecency       |
+------------------+---------+--------------------+------------------+------------------+-----------------+
|Champions         |1345     |1.2213762754999993E7|9080.86450185873  |16.610408921933086|19.30557620817844|
|Loyal             |721      |1911016.986         |2650.5089958391122|7.680998613037448 |76.31345353675451|
|At Risk           |708      |1563985.7519999996  |2209.019423728813 |5.427966101694915 |360.4901129943503|
|Lost/Other        |2343     |1343799.4349999994  |573.5379577464786 |1.5795988049509175|351.8506188647034|
|Potential Loyalist|761      |710864.2500000001   |934.1185939553221 |2.031537450722733 |26.40473061760841|
+------------------+---------+--------------------+------------------+------------------+-----------------+



In [32]:
from pyspark.sql import functions as F

# 1. Lưu kết quả gom nhóm vào một DataFrame (thay vì chỉ .show() luôn như code gốc)
segment_summary = rfm_segments.groupBy('Segment').agg(
    F.count('*').alias('Customers'),
    F.sum('Monetary').alias('Revenue'),
    F.avg('Monetary').alias('AvgMonetary'),
    F.avg('Frequency').alias('AvgFrequency'),
    F.avg('Recency').alias('AvgRecency')
)

# 2. Lấy tổng số lượng khách hàng và tổng doanh thu toàn tệp
total_customers = rfm_segments.count()
total_revenue = rfm_segments.agg(F.sum('Monetary')).first()[0]

# 3. Tính toán tỷ trọng % (Percentage) và làm tròn
segment_summary_pct = (segment_summary
    .withColumn('Cust_Pct (%)', F.round((F.col('Customers') / F.lit(total_customers)) * 100, 1))
    .withColumn('Rev_Pct (%)', F.round((F.col('Revenue') / F.lit(total_revenue)) * 100, 1))
    .orderBy(F.desc('Revenue')))

# 4. Hiển thị bảng kết quả với tỷ trọng
segment_summary_pct.select(
    'Segment',
    'Customers',
    'Cust_Pct (%)',
    'Revenue',
    'Rev_Pct (%)',
    'AvgFrequency',
    'AvgRecency'
).show(truncate=False)

+------------------+---------+------------+--------------------+-----------+------------------+-----------------+
|Segment           |Customers|Cust_Pct (%)|Revenue             |Rev_Pct (%)|AvgFrequency      |AvgRecency       |
+------------------+---------+------------+--------------------+-----------+------------------+-----------------+
|Champions         |1345     |22.9        |1.2213762754999993E7|68.8       |16.610408921933086|19.30557620817844|
|Loyal             |721      |12.3        |1911016.986         |10.8       |7.680998613037448 |76.31345353675451|
|At Risk           |708      |12.0        |1563985.7519999996  |8.8        |5.427966101694915 |360.4901129943503|
|Lost/Other        |2343     |39.9        |1343799.4349999994  |7.6        |1.5795988049509175|351.8506188647034|
|Potential Loyalist|761      |12.9        |710864.2500000001   |4.0        |2.031537450722733 |26.40473061760841|
+------------------+---------+------------+--------------------+-----------+------------

In [33]:
import plotly.express as px
import pandas as pd

# 1. Convert the PySpark summary DataFrame to a Pandas DataFrame for plotting
# Note: It is safe to use .toPandas() here because the summary table is very small (only a few rows)
df_plot = segment_summary_pct.toPandas()

# =====================================================================
# CHART 1: The "80/20 Rule" (Customer % vs Revenue %)
# Visually compares how much of your base a segment takes up vs. how much they pay
# =====================================================================

# Melt the dataframe so Plotly can easily group the bars side-by-side
# Convert to Pandas and melt
df_plot = segment_summary_pct.toPandas()

df_melted = df_plot.melt(
    id_vars='Segment',
    value_vars=['Cust_Pct (%)', 'Rev_Pct (%)'],
    var_name='Metric',
    value_name='Percentage'
)

df_melted['Metric'] = df_melted['Metric'].replace({
    'Cust_Pct (%)': '% of Customers',
    'Rev_Pct (%)': '% of Revenue'
})

# Define your exact desired order
rfm_order = ["Champions", "Loyal", "Potential Loyalist", "At Risk", "Lost/Other"]

fig1 = px.bar(
    df_melted,
    x='Segment',
    y='Percentage',
    color='Metric',
    barmode='group',
    text_auto='.1f',
    title='Who Pays the Rent? (Customer Share vs. Revenue Share)',
    color_discrete_sequence=['#636EFA', '#00CC96'],
    # 👇 THIS IS THE MAGIC LINE 👇
    category_orders={'Segment': rfm_order}
)

fig1.update_layout(yaxis_title="Percentage (%)", xaxis_title="RFM Segment")
fig1.show()

# =====================================================================
# CHART 2: The "Relationship Status" Bubble Chart (Risk vs. Reward)
# Shows Recency, Frequency, and Revenue in one clean view
# =====================================================================

fig2 = px.scatter(
    df_plot,
    x='AvgRecency',
    y='AvgFrequency',
    size='Revenue',
    color='Segment',
    hover_name='Segment',
    size_max=50, # Scales the bubbles nicely
    title='Customer Relationship Status (Bubble size = Total Revenue)',
    labels={
        'AvgRecency': 'Days Since Last Purchase (Lower is Better)',
        'AvgFrequency': 'Average Number of Purchases'
    }
)

# Reverse the X-axis so "Recent" (smaller numbers) is on the right side
# This puts your "Champions" (High Frequency, High Recency) in the top right corner
fig2.update_xaxes(autorange="reversed")

# Add funny quadrant annotations (Optional, but adds flavor)
fig2.add_annotation(x=10, y=df_plot['AvgFrequency'].max(), text="❤️ VIP", showarrow=False, font=dict(size=14, color="green"))
#fig2.add_annotation(x=df_plot['AvgRecency'].max(), y=df_plot['AvgFrequency'].max(), text="💔 The Exes", showarrow=False, font=dict(size=14, color="orange"))
fig2.add_annotation(x=df_plot['AvgRecency'].max(), y=df_plot['AvgFrequency'].min(), text="👻 Ghosted Us", showarrow=False, font=dict(size=14, color="red"))

fig2.update_layout(template="plotly_white") # Clean white background
fig2.show()

**Đề xuất một hành động tiếp thị (marketing action) phù hợp cho từng nhóm:**

*   Champions (1.345 khách - 22.9% tệp khách, nhưng tạo ra £12.21M $\approx$ 68.8% doanh thu): Mua cực kỳ thường xuyên (TB 16.6 lần), mới mua gần đây (~19 ngày) và chi tiêu khủng (TB £9,080). Hành động: Chăm sóc VIP 1-1 (Account Manager riêng), chiết khấu theo sản lượng (volume discount), ưu tiên giữ hàng trước vào mùa cao điểm Giáng sinh.   

*   Loyal (721 khách - £1.91M): Mua đều (TB 7.7 lần), chi tiêu khá (£2,650). Hành động: Gợi ý bán chéo (Cross-sell / Up-sell) các sản phẩm mới để đẩy họ lên nhóm Champions.   

*   Potential Loyalist (761 khách - £0.71M): Mới mua gần đây (26 ngày) nhưng số lần mua còn ít (2 lần). Hành động: Chương trình khách hàng thân thiết (Loyalty onboarding), tặng ưu đãi cho lần mua thứ 3 và thứ 4.   

*   At Risk (708 khách - £1.56M): Từng mua nhiều lần (TB 5.4 lần) và mang lại giá trị cao (TB £2,209) nhưng đã gần 1 năm chưa quay lại (AvgRecency = 360.5 ngày). Hành động: Nhóm cần ưu tiên giữ chân/tái kích hoạt khẩn cấp bằng chiến dịch Win-back (gửi catalog mùa lễ hội mới kèm voucher ưu đãi đặc biệt cho đơn đặt hàng lại).   

*   Lost/Other (2.343 khách - đông nhất ~39.8% nhưng chỉ tạo ra £1.34M): Khách vãng lai mua 1–2 lần từ rất lâu (351 ngày), giá trị thấp (£573). Hành động: Chỉ gửi email marketing tự động chi phí thấp, không dồn ngân sách lớn.




# Phần E: Spark SQL

## Bài 10. Truy vấn SQL

Tạo temporary view sales và viết ít nhất 6 truy vấn Spark SQL:

34. Doanh thu theo tháng.
35. Top 10 sản phẩm theo Revenue.
36. Top 10 khách hàng theo Revenue.
37. Top 10 quốc gia ngoài UK.
38. Các tháng có AOV cao nhất.
39. Sản phẩm xuất hiện trong nhiều hóa đơn nhất.
```python
sales.createOrReplaceTempView('sales')
spark.sql('SELECT ... FROM sales GROUP BY ...')
```

In [34]:
# Đăng ký DataFrame sales thành temporary view
sales.createOrReplaceTempView('sales')

# Chạy truy vấn Spark SQL
spark.sql('''
SELECT YearMonth,
       ROUND(SUM(Revenue), 2) AS Revenue,
       COUNT(DISTINCT InvoiceNo) AS Orders
FROM sales
GROUP BY YearMonth
ORDER BY YearMonth
''').show(30)

+---------+----------+------+
|YearMonth|   Revenue|Orders|
+---------+----------+------+
|  2009-12| 825685.76|  1682|
|  2010-01|  652708.5|  1105|
|  2010-02| 553339.74|  1201|
|  2010-03| 833570.13|  1681|
|  2010-04| 681528.99|  1462|
|  2010-05| 659858.86|  1500|
|  2010-06| 752270.14|  1645|
|  2010-07| 650712.94|  1529|
|  2010-08| 697274.91|  1425|
|  2010-09| 924333.01|  1839|
|  2010-10|1165483.91|  2301|
|  2010-11|1470272.48|  2747|
|  2010-12|1262598.79|  1559|
|  2011-01| 691364.56|  1086|
|  2011-02| 523631.89|  1100|
|  2011-03| 717639.36|  1454|
|  2011-04| 537808.62|  1246|
|  2011-05| 770536.02|  1681|
|  2011-06|  761739.9|  1533|
|  2011-07| 719221.19|  1475|
|  2011-08| 759138.38|  1361|
|  2011-09|1058590.17|  1837|
|  2011-10| 1154979.3|  2040|
|  2011-11|1509496.33|  2769|
|  2011-12| 638810.68|   819|
+---------+----------+------+



# Phần F: Câu hỏi phân tích kinh doanh

Không chỉ chạy code. Anh/chị phải trả lời tối thiểu 6 câu hỏi sau bằng số liệu/biểu đồ:

- Xu hướng doanh thu theo thời gian như thế nào? Có dấu hiệu mùa vụ hay không?
- Sản phẩm nào thực sự quan trọng nếu xét đồng thời Revenue, Quantity và số hóa đơn?
- Thị trường ngoài UK nào có tiềm năng nhất?
- Doanh thu có tập trung vào một số ít khách hàng hay không?
- Nhóm RFM nào tạo ra phần lớn doanh thu?
- Nhóm khách hàng nào nên được ưu tiên giữ chân?
- Tỷ lệ hủy giao dịch có đáng kể hay không và có tập trung theo sản phẩm/thị trường?


In [35]:
print("--- 35. Top 10 sản phẩm theo Revenue ---")
spark.sql('''
SELECT StockCode, Description, ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM sales
GROUP BY StockCode, Description
ORDER BY Total_Revenue DESC
LIMIT 10
''').show(truncate=False)

print("\n--- 36. Top 10 khách hàng theo Revenue ---")
spark.sql('''
SELECT CustomerID, ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM sales
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 10
''').show(truncate=False)

print("\n--- 37. Top 10 quốc gia ngoài UK theo Revenue ---")
spark.sql('''
SELECT Country, ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM sales
WHERE Country != 'United Kingdom'
GROUP BY Country
ORDER BY Total_Revenue DESC
LIMIT 10
''').show(truncate=False)

print("\n--- 38. Các tháng có AOV cao nhất ---")
spark.sql('''
SELECT YearMonth,
       ROUND(SUM(Revenue), 2) AS Total_Revenue,
       COUNT(DISTINCT InvoiceNo) AS Total_Orders,
       ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS AOV
FROM sales
GROUP BY YearMonth
ORDER BY AOV DESC
''').show(30, truncate=False)

print("\n--- 39. Sản phẩm xuất hiện trong nhiều hóa đơn nhất ---")
spark.sql('''
SELECT StockCode, Description, COUNT(DISTINCT InvoiceNo) AS Invoice_Count
FROM sales
GROUP BY StockCode, Description
ORDER BY Invoice_Count DESC
LIMIT 10
''').show(truncate=False)

--- 35. Top 10 sản phẩm theo Revenue ---
+---------+----------------------------------+-------------+
|StockCode|Description                       |Total_Revenue|
+---------+----------------------------------+-------------+
|22423    |REGENCY CAKESTAND 3 TIER          |344563.25    |
|M        |Manual                            |340716.28    |
|DOT      |DOTCOM POSTAGE                    |322657.48    |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|262931.16    |
|23843    |PAPER CRAFT , LITTLE BIRDIE       |168469.6     |
|47566    |PARTY BUNTING                     |149187.05    |
|85099B   |JUMBO BAG RED RETROSPOT           |148823.92    |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |132187.92    |
|POST     |POSTAGE                           |127597.42    |
|22086    |PAPER CHAIN KIT 50'S CHRISTMAS    |123141.54    |
+---------+----------------------------------+-------------+


--- 36. Top 10 khách hàng theo Revenue ---
+----------+-------------+
|CustomerID|Total_Revenue|
+-----

## Bài 11. Phân tích Pareto 80/20

* Sắp xếp sản phẩm theo Revenue giảm dần, tính cumulative revenue và cumulative revenue share.
* Xác định bao nhiêu % sản phẩm tạo ra 80% doanh thu.
* Lặp lại cho khách hàng.

Câu hỏi: Quy luật 80/20 có xuất hiện trong dữ liệu này hay không? Không được kết luận trước khi tính
toán.


In [36]:
total_product_revenue = product_sales.agg(F.sum('Revenue')).first()[0]
w = Window.orderBy(F.desc('Revenue')).rowsBetween(Window.unboundedPreceding, Window.currentRow)
pareto_products   = (product_sales
    .orderBy(F.desc('Revenue'))
    .withColumn('CumRevenue', F.sum('Revenue').over(w))
    .withColumn('CumShare', F.col('CumRevenue') / F.lit(total_product_revenue)))
pareto_products.show(20, truncate=False)

+---------+-----------------------------------+--------+-----+------+----------+--------------------+
|StockCode|Description                        |Revenue |Units|Orders|CumRevenue|CumShare            |
+---------+-----------------------------------+--------+-----+------+----------+--------------------+
|22423    |REGENCY CAKESTAND 3 TIER           |344563.0|27577|3918  |344563.0  |0.016429144044565813|
|M        |Manual                             |340716.0|10050|784   |685279.0  |0.032674858884198285|
|DOT      |DOTCOM POSTAGE                     |322657.0|1436 |1415  |1007936.0 |0.048059500676809426|
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER |262931.0|96086|5356  |1270867.0 |0.06059634088536848 |
|23843    |PAPER CRAFT , LITTLE BIRDIE        |168470.0|80995|1     |1439337.0 |0.06862917638189016 |
|47566    |PARTY BUNTING                      |149187.0|28378|2674  |1588524.0 |0.07574257716078006 |
|85099B   |JUMBO BAG RED RETROSPOT            |148824.0|78860|3245  |1737348.0 |0.

In [37]:
# Pareto cho Sản phẩm
total_products = pareto_products.count()
products_80 = pareto_products.filter(F.col('CumShare') <= 0.80).count() + 1
print(f"Tỷ lệ sản phẩm tạo ra 80% doanh thu: {products_80}/{total_products} ({products_80/total_products*100:.2f}%)")

# Pareto cho Khách hàng
cust_80 = cust_ranked.filter(F.col('CumShare') <= 0.80).count() + 1
print(f"Tỷ lệ khách hàng tạo ra 80% doanh thu: {cust_80}/{total_cust} ({cust_80/total_cust*100:.2f}%)")

Tỷ lệ sản phẩm tạo ra 80% doanh thu: 1152/5630 (20.46%)
Tỷ lệ khách hàng tạo ra 80% doanh thu: 1353/5878 (23.02%)


Câu hỏi: Quy luật 80/20 có xuất hiện trong dữ liệu này hay không? Không được kết luận trước khi tính toán.

Khoảng ~21–23% số lượng sản phẩm tạo ra 80% tổng doanh thu, và khoảng ~23–24% số lượng khách hàng tạo ra 80% doanh thu định danh. Như vậy, quy luật Pareto 80/20 thực sự xuất hiện và phản ánh rất sát thực tế trên cả hai chiều Sản phẩm và Khách hàng của doanh nghiệp này!

# Câu hỏi thảo luận cuối bài

40. Vì sao Revenue phải được tạo từ Quantity × UnitPrice thay vì chỉ phân tích Quantity?
41. Tại sao không nên collect() hoặc `toPandas()` toàn bộ hơn một triệu dòng?
42. Tại sao phải tách phân tích sales_clean và sales_customer?
43. Việc loại giao dịch hủy có thể làm mất thông tin kinh doanh nào?
44. RFM mô tả giá trị khách hàng ở khía cạnh nào và còn thiếu thông tin gì?
45. Nếu dữ liệu tăng lên hàng trăm triệu dòng, phần nào của notebook vẫn giữ nguyên về tư duy phân
tích?

**40. Vì sao dùng Revenue = Quantity × UnitPrice thay vì chỉ phân tích Quantity?**

Vì số lượng bán ra không phản ánh đầy đủ giá trị kinh doanh của sản phẩm.

Ví dụ:

* Bán 10.000 sản phẩm, mỗi sản phẩm giá £0,20 → doanh thu £2.000.
* Bán 500 sản phẩm, mỗi sản phẩm giá £15 → doanh thu £7.500.

Mặc dù sản phẩm thứ nhất bán được nhiều hơn, sản phẩm thứ hai lại tạo ra doanh thu cao hơn.

Vì vậy, cần phân tích cả **Revenue (doanh thu), Quantity (số lượng bán) và Orders (số đơn hàng)** để hiểu rõ hơn hiệu quả kinh doanh của từng sản phẩm.

**41. Tại sao không nên dùng `collect()` hoặc `toPandas()` cho toàn bộ dữ liệu?**

Spark được thiết kế để xử lý dữ liệu trên nhiều máy tính hoặc nhiều tiến trình.

* `collect()` đưa toàn bộ kết quả về Driver.
* `toPandas()` chuyển dữ liệu Spark thành DataFrame của Pandas trên Driver.

Nếu dữ liệu quá lớn, Driver có thể hết RAM, khiến chương trình bị lỗi. Việc chuyển một lượng lớn dữ liệu qua mạng cũng làm giảm hiệu quả xử lý.

Vì vậy, **nên để Spark xử lý dữ liệu và chỉ chuyển kết quả nhỏ cần thiết sang Pandas** để vẽ biểu đồ hoặc phân tích thêm.

**42. Tại sao phải tách `sales_clean` và `sales_customer`?**

Hai DataFrame này phục vụ các mục đích phân tích khác nhau.

* **`sales_clean`:** Giữ lại các giao dịch hợp lệ cho những phân tích như doanh thu theo tháng, quốc gia và sản phẩm. Nếu loại bỏ ngay các dòng không có `CustomerID`, tổng doanh thu có thể bị thiếu.
* **`sales_customer`:** Chỉ giữ các giao dịch có `CustomerID` hợp lệ để phân tích hành vi mua hàng của từng khách và xây dựng mô hình RFM.

Ngoài ra, cần nhận diện và giữ thông tin về giao dịch hủy hoặc trả hàng thay vì xóa chúng ngay từ đầu. Những thông tin này giúp phân tích tình trạng hủy đơn, trả hàng và xác định các sản phẩm có vấn đề.

**Lưu ý:** Giao dịch hủy và giao dịch trả hàng không hoàn toàn giống nhau. Cần xác định đúng loại giao dịch trước khi tính tỷ lệ hủy hoặc trả hàng.

**43. RFM có hạn chế gì?**

RFM đánh giá khách hàng dựa trên ba yếu tố:

* **Recency (R):** Khách hàng mua hàng gần đây hay đã lâu chưa mua?
* **Frequency (F):** Khách hàng mua hàng bao nhiêu lần?
* **Monetary (M):** Khách hàng đã chi tiêu bao nhiêu?

Tuy nhiên, RFM có ba hạn chế chính:

1. **Chưa phản ánh lợi nhuận:** Khách hàng chi tiêu nhiều chưa chắc mang lại lợi nhuận cao nếu họ thường mua hàng giảm giá hoặc trả lại nhiều sản phẩm.
2. **Chưa phản ánh đầy đủ đặc điểm mua hàng:** Khách hàng lâu năm hoặc chỉ mua theo mùa có thể bị đánh giá thấp nếu khoảng cách giữa các lần mua lớn.
3. **Chưa dự báo hành vi tương lai:** RFM chủ yếu mô tả giao dịch trong quá khứ, chưa cho biết chắc chắn khách hàng có tiếp tục mua hàng hay sẽ rời bỏ doanh nghiệp.

Vì vậy, RFM là công cụ phân nhóm khách hàng hữu ích, nhưng nên kết hợp với dữ liệu về lợi nhuận, thời gian gắn bó và mô hình dự báo nếu muốn đánh giá giá trị khách hàng toàn diện hơn.

**44. Nếu dữ liệu tăng lên 100 triệu dòng, Spark có lợi thế gì và tư duy phân tích nào vẫn giữ nguyên?**

**Lợi thế của Spark:**

Spark có thể phân chia công việc trên nhiều máy tính, giúp xử lý dữ liệu lớn hiệu quả hơn khi được cung cấp đủ tài nguyên. Spark cũng tối ưu hóa kế hoạch thực thi để giảm bớt các thao tác không cần thiết.

Tuy nhiên, hiệu quả thực tế vẫn phụ thuộc vào dung lượng dữ liệu, bộ nhớ, số máy và cách viết chương trình.

**Tư duy phân tích vẫn giữ nguyên:**

Quy trình cơ bản không thay đổi:

1. Làm sạch dữ liệu.
2. Tạo các biến cần thiết.
3. Tổng hợp dữ liệu bằng `groupBy`.
4. Sử dụng Window Functions để phân tích RFM và Pareto.
5. Chỉ chuyển kết quả nhỏ cần thiết sang Pandas.

Điều cần thay đổi khi dữ liệu lớn hơn là cách tối ưu chương trình và phân bổ tài nguyên, chứ không nhất thiết phải thay đổi toàn bộ logic phân tích.

**45. Nếu có thêm dữ liệu Cost và Profit, bài phân tích nên thay đổi thế nào?**

Khi có dữ liệu chi phí, ta có thể đánh giá hiệu quả kinh doanh chính xác hơn thay vì chỉ nhìn vào doanh thu.

Hai chỉ tiêu quan trọng là:

* **Profit (lợi nhuận):** Revenue − Cost, nếu Cost phản ánh đầy đủ các chi phí cần tính.
* **Profit Margin (biên lợi nhuận):** Profit / Revenue × 100%.

Khi đó, ta có thể mở rộng phân tích theo ba hướng:

1. **Phân tích sản phẩm:** Xác định sản phẩm tạo ra doanh thu cao nhưng lợi nhuận thấp.
2. **Phân tích khách hàng:** Tìm ra nhóm khách hàng mang lại lợi nhuận lớn, thay vì chỉ chi tiêu nhiều.
3. **Phân tích Pareto:** Xác định nhóm sản phẩm hoặc khách hàng đóng góp phần lớn tổng lợi nhuận.

Ví dụ, một sản phẩm có doanh thu cao nhưng chi phí cũng rất cao có thể mang lại ít lợi nhuận hơn một sản phẩm bán được ít hơn.

**Kết luận:** Bổ sung dữ liệu chi phí giúp doanh nghiệp chuyển từ câu hỏi “Bán được bao nhiêu?” sang câu hỏi quan trọng hơn: “Thực sự kiếm được bao nhiêu?”

# Section G: Integrated Findings, Insights, and Recommendations

## Bảng điểm đối chiếu toàn cảnh (Integrated Executive Scorecard)

Tập dữ liệu **UCI Online Retail II** bao gồm 1.067.371 dòng bản ghi giao dịch trong giai đoạn 01/12/2009–09/12/2011. Sau khi lọc bỏ các hóa đơn hủy và các dòng có số lượng hoặc đơn giá không dương, tập giao dịch bán hàng hợp lệ (`sales_clean`, gồm 1.041.670 dòng) ghi nhận **tổng doanh thu gộp (Gross Valid Revenue) đạt £20.972.594,57** trên **40.077 đơn hàng** và **11.420.305 đơn vị sản phẩm**.

Tuy nhiên, việc đánh giá hiệu quả kinh doanh thuần túy dựa trên doanh thu gộp dễ dẫn đến các quyết định sai lệch. Khi đối chiếu chéo với các chứng từ ghi giảm/hủy đơn (`IsCancelled = True`), bóc tách nhóm mã phí hệ thống phi hàng hóa, và phân rã theo tệp khách hàng, cấu trúc vận hành thực sự của doanh nghiệp lộ diện qua năm quy luật kinh tế dưới đây.

**Định hướng điều hành cốt lõi:** Doanh nghiệp cần chuyển dịch hệ thống báo cáo quản trị từ **Doanh thu gộp danh nghĩa (Gross Revenue)** sang **Doanh thu ròng hàng hóa đã đối trừ (Reconciled Net Merchandise Revenue)**; chuẩn hóa danh mục SKU trước khi lập kế hoạch tồn kho Quý 4; đồng thời thiết kế chính sách giữ chân khách hàng và chiết khấu xuất khẩu dựa trên kinh tế học đơn vị (Unit Economics).

In [38]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql import functions as F, Window

# Tái sử dụng các biến từ pipeline trước: sales, df, NON_MERCH_CODES
NON_MERCH_CODES = ['M', 'DOT', 'POST', 'B', 'D', 'CRUK', 'BANK CHARGES', 'PADS']

sales_enriched = (sales
    .withColumn('IsMerchandise', ~F.col('StockCode').isin(NON_MERCH_CODES))
    .withColumn('CustomerChannel',
        F.when(F.col('CustomerID').isNotNull(), 'Identified B2B (Có CustomerID)')
         .otherwise('Unidentified / Guest (Thiếu CustomerID)')
    )
)

sales_cancelled = (df
    .filter(F.col('IsCancelled') & (F.col('UnitPrice') > 0))
    .withColumn('CancelledRevenue', F.abs(F.col('Quantity')) * F.col('UnitPrice'))
    .withColumn('CancelledUnits', F.abs(F.col('Quantity')))
    .withColumn('YearMonth', F.date_format('InvoiceDate', 'yyyy-MM'))
    .withColumn('IsMerchandise', ~F.col('StockCode').isin(NON_MERCH_CODES))
)

gross_total = float(sales_enriched.agg(F.sum('Revenue')).first()[0])
orders = sales_enriched.select('InvoiceNo').distinct().count()

cancel_stats = sales_cancelled.agg(
    F.count('*').alias('Cancel_Lines'),
    F.countDistinct('InvoiceNo').alias('Cancel_Orders'),
    F.sum('CancelledRevenue').alias('Cancel_Rev')
).first()
cancel_total = float(cancel_stats['Cancel_Rev'])

non_merch_gross = float(sales_enriched.filter(~F.col('IsMerchandise')).agg(F.sum('Revenue')).first()[0])
non_merch_cancel = float(sales_cancelled.filter(~F.col('IsMerchandise')).agg(F.sum('CancelledRevenue')).first()[0])
merch_net_total = (gross_total - non_merch_gross) - (cancel_total - non_merch_cancel)

print("=== BẢNG ĐỐI CHIẾU DOANH THU GỘP, HOÀN HỦY VÀ DOANH THU RÒNG HÀNG HÓA ===")
scorecard_pdf = pd.DataFrame([
    {"Chỉ tiêu (Metric)": "1. Tổng Doanh thu Gộp hợp lệ", "Giá trị (£)": f"£{gross_total:,.2f}", "Tỷ trọng / Ghi chú": f"{orders:,} đơn hàng | AOV = £{gross_total/orders:,.2f}"},
    {"Chỉ tiêu (Metric)": "2. Tổng Giá trị Giao dịch Hủy", "Giá trị (£)": f"-£{cancel_total:,.2f}", "Tỷ trọng / Ghi chú": f"{cancel_stats['Cancel_Orders']:,} đơn hủy | Bằng {cancel_total/gross_total*100:.2f}% Gross Revenue"},
    {"Chỉ tiêu (Metric)": "3. Doanh thu Ròng Toàn sàn", "Giá trị (£)": f"£{gross_total - cancel_total:,.2f}", "Tỷ trọng / Ghi chú": "Gross Valid Revenue trừ đi tổng giá trị đơn hủy"},
    {"Chỉ tiêu (Metric)": "4. Doanh thu Gộp từ Mã phí hệ thống", "Giá trị (£)": f"£{non_merch_gross:,.2f}", "Tỷ trọng / Ghi chú": f"Bị hủy đối ứng -£{non_merch_cancel:,.2f}"},
    {"Chỉ tiêu (Metric)": "5. Doanh thu Ròng Hàng hóa Vật lý", "Giá trị (£)": f"£{merch_net_total:,.2f}", "Tỷ trọng / Ghi chú": "Thước đo chuẩn xác nhất về tiêu thụ thực chất"}
])
display(scorecard_pdf)

=== BẢNG ĐỐI CHIẾU DOANH THU GỘP, HOÀN HỦY VÀ DOANH THU RÒNG HÀNG HÓA ===


,Chỉ tiêu (Metric),Giá trị (£),Tỷ trọng / Ghi chú
0,1. Tổng Doanh thu Gộp hợp lệ,"£20,972,594.57","40,077 đơn hàng | AOV = £523.31"
1,2. Tổng Giá trị Giao dịch Hủy,"-£1,527,415.00","8,292 đơn hủy | Bằng 7.28% Gross Revenue"
2,3. Doanh thu Ròng Toàn sàn,"£19,445,179.57",Gross Valid Revenue trừ đi tổng giá trị đơn hủy
3,4. Doanh thu Gộp từ Mã phí hệ thống,"£802,965.39","Bị hủy đối ứng -£497,475.50"
4,5. Doanh thu Ròng Hàng hóa Vật lý,"£19,139,689.68",Thước đo chuẩn xác nhất về tiêu thụ thực chất


## G.1. Phát hiện 1 — Chu kỳ tích trữ hàng Quý 4, nhịp giao dịch và kênh khách hàng

**Cơ chế trễ chuỗi cung ứng B2B (Lead-time Seasonality):** Trong ngành hàng quà tặng, các cửa hàng bán lẻ (independent gift shops) phải nhập hàng từ tháng 9 đến tháng 11 để kịp trưng bày trước mùa mua sắm Giáng sinh. Sự cố đơn hàng `23843` ngày 09/12/2011 chứng minh rằng nếu chỉ nhìn vào AOV gộp tháng 12 mà không loại trừ giao dịch hủy, nhà phân tích sẽ đánh giá sai sức cầu thực tế cuối kỳ.

**Tập trung giao dịch vào tuần làm việc:** Các ngày làm việc từ thứ Hai đến thứ Sáu chiếm **91,15% tổng doanh thu gộp**, với 79,10% doanh thu diễn ra từ **10h00 đến 15h00**. Ngày thứ Bảy gần như không phát sinh hoạt động (0,047% doanh thu).

**Bản chất của tệp giao dịch khuyết CustomerID:** Không phải là khách lẻ. Trung bình mỗi đơn khuyết ID chứa **76 dòng hàng**, AOV **£1.038,99**. Đây là sự kết hợp của các bảng kê tổng hợp đơn hàng Dotcom (mã `DOT`) và đơn mua sỉ không mở tài khoản công nợ.

*Khuyến nghị:* Lập kế hoạch tồn kho Quý 4 phải hoàn tất trước 01/09. Tối ưu nhân sự kho vận vào thứ Ba-thứ Năm, bảo trì hệ thống vào thứ Bảy.

In [39]:
monthly = sales_enriched.groupBy('YearMonth').agg(F.sum('Revenue').alias('Revenue'), (F.sum('Revenue')/F.countDistinct('InvoiceNo')).alias('AOV'))
monthly_cancel = sales_cancelled.groupBy('YearMonth').agg(F.round(F.sum('CancelledRevenue'), 2).alias('Cancelled_Revenue'))

pdf_monthly_merged = (monthly
    .join(monthly_cancel, on='YearMonth', how='left')
    .na.fill({'Cancelled_Revenue': 0.0})
    .withColumn('Net_Revenue', F.round(F.col('Revenue') - F.col('Cancelled_Revenue'), 2))
    .orderBy('YearMonth')
    .toPandas()
)

dow_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot_dow_hour = (sales_enriched
    .groupBy('DayOfWeek', 'Hour')
    .agg(F.round(F.sum('Revenue') / 1000.0, 1).alias('Revenue_k'))
    .toPandas()
    .pivot(index='DayOfWeek', columns='Hour', values='Revenue_k')
    .reindex(dow_order)
    .fillna(0.0)
)

fig_g1 = make_subplots(
    rows=1, cols=2, column_widths=[0.56, 0.44],
    specs=[[{"secondary_y": True}, {"secondary_y": False}]],
    subplot_titles=("A. Sóng Mùa vụ Tháng (Gross vs. Net)", "B. Doanh thu theo Thứ & Giờ (£k)")
)

fig_g1.add_trace(go.Bar(x=pdf_monthly_merged['YearMonth'], y=pdf_monthly_merged['Revenue'], name='Gross Revenue', marker_color='#636EFA'), row=1, col=1, secondary_y=False)
fig_g1.add_trace(go.Scatter(x=pdf_monthly_merged['YearMonth'], y=pdf_monthly_merged['Net_Revenue'], name='Net Revenue', mode='lines+markers', line=dict(color='#00CC96', width=2.5, dash='dot')), row=1, col=1, secondary_y=False)
fig_g1.add_trace(go.Scatter(x=pdf_monthly_merged['YearMonth'], y=pdf_monthly_merged['AOV'], name='AOV (£/Đơn)', mode='lines+markers', line=dict(color='#EF553B')), row=1, col=1, secondary_y=True)

fig_g1.add_trace(go.Heatmap(z=pivot_dow_hour.values, x=[f"{h:02d}:00" for h in pivot_dow_hour.columns], y=pivot_dow_hour.index, colorscale='YlGnBu'), row=1, col=2)

fig_g1.update_layout(title_text="Hình G.1: Mùa vụ Quý 4 và Nhịp giao dịch Giờ Hành Chính", height=450, template='plotly_white')
fig_g1.show()

## G.2. Phát hiện 2 — Sản phẩm cốt lõi: Phễu lưu lượng và Cỗ máy lợi nhuận

Doanh thu tuân theo quy luật tập trung Pareto: **20,46% mã hàng (1.152 SKU)** tạo ra **80% tổng doanh thu**. Tuy nhiên, bảng xếp hạng Gross Revenue bị biến dạng nặng nề bởi mã phí (`M`, `DOT`) và các đơn hàng đột biến bị hủy (`23843`, `23166`).

Sau khi gom nhóm chính xác theo `StockCode` (khắc phục lỗi phân mảnh chuỗi mô tả) và khấu trừ hoàn hủy, sự phân hóa chiến lược trong giỏ hàng lộ diện:
*   **Cash Cows (Sản phẩm tạo doanh thu):** Mã `22423` (Khay bánh 3 tầng) dẫn đầu tuyệt đối với **Doanh thu ròng £327.813**.
*   **Traffic Builders (Sản phẩm tạo sản lượng):** Mã `84077` (Mô hình máy bay) dẫn đầu về lượng bán (**110.138 chiếc**) nhưng giá rẻ (£0.23/chiếc), tạo ra doanh thu thấp. Mã `85123A` (Giá nến) và `21212` (Khuôn bánh) xuất hiện trong hàng ngàn giỏ hàng, đóng vai trò mồi câu thu hút khách mua sỉ.

*Khuyến nghị:* Áp dụng chiến lược "Bundling" (bán theo gói) bằng cách ghép các mặt hàng Traffic Builders với Cash Cows để nâng biên lợi nhuận gộp.

In [40]:
# Tái cấu trúc chuẩn xác Exact Orders và Net Revenue
prod_exact = (sales_enriched
    .groupBy('StockCode')
    .agg(
        F.first('Description', ignorenulls=True).alias('Description'),
        F.sum('Revenue').alias('Gross_Revenue'),
        F.sum('Quantity').alias('Gross_Units'),
        F.countDistinct('InvoiceNo').alias('Exact_Orders')
    )
    .withColumn('IsMerchandise', ~F.col('StockCode').isin(NON_MERCH_CODES))
)

prod_cancel_clean = sales_cancelled.groupBy('StockCode').agg(F.sum('CancelledRevenue').alias('Cancelled_Revenue'))

prod_reconciled = (prod_exact
    .join(prod_cancel_clean, on='StockCode', how='left')
    .na.fill({'Cancelled_Revenue': 0.0})
    .withColumn('Net_Revenue', F.col('Gross_Revenue') - F.col('Cancelled_Revenue'))
    .withColumn('Cancel_Pct', F.when(F.col('Gross_Revenue') == 0, 0.0).otherwise(F.col('Cancelled_Revenue') / F.col('Gross_Revenue') * 100))
    .withColumn('SKU_Category',
        F.when(~F.col('IsMerchandise'), '1. Non-Merchandise')
         .when(F.col('Cancel_Pct') >= 50.0, '2. Anomaly')
         .otherwise('3. True Hero SKU')
    )
)

merch_only = prod_reconciled.filter(F.col('SKU_Category') == '3. True Hero SKU')
pdf_top_rev = merch_only.orderBy(F.desc('Net_Revenue')).limit(10).toPandas().sort_values('Net_Revenue')
pdf_top_qty = merch_only.orderBy(F.desc('Gross_Units')).limit(10).toPandas().sort_values('Gross_Units')

fig_g2 = make_subplots(rows=1, cols=2, subplot_titles=("A. Top 10 SKU theo Net Revenue (£)", "B. Top 10 SKU theo Số lượng bán (Units)"), horizontal_spacing=0.3)
fig_g2.add_trace(go.Bar(y=pdf_top_rev['Description'].str.slice(0, 25), x=pdf_top_rev['Net_Revenue'], orientation='h', marker_color='#00CC96', name='Net Revenue'), row=1, col=1)
fig_g2.add_trace(go.Bar(y=pdf_top_qty['Description'].str.slice(0, 25), x=pdf_top_qty['Gross_Units'], orientation='h', marker_color='#636EFA', name='Gross Units'), row=1, col=2)

fig_g2.update_layout(title_text="Hình G.2: Sự lệch pha giữa Cỗ máy Doanh thu và Phễu Sản lượng (Traffic Builders)", height=400, showlegend=False, template='plotly_white')
fig_g2.show()

## G.3. Phát hiện 3 — Rủi ro tập trung RFM: Loại bỏ "VIP Ảo" và thu hồi phân khúc At Risk

**Kiểm toán VIP Ảo:** Việc chấm điểm RFM dựa trên Doanh thu Gộp (Gross) đã xếp hạng nhầm các tài khoản hủy toàn bộ đơn (như `16446` và `12346`) vào nhóm VIP. Sau khi khấu trừ đơn hủy, doanh thu ròng của các mã này bằng 0.

**Tập trung cực đoan (Pareto):** 23,02% khách hàng định danh tạo ra 80% tổng doanh thu ròng. Nhóm **Top 1% (58 khách hàng)** nắm giữ tới **31.8%** doanh thu (sự phụ thuộc đại lý lớn).
Phân khúc *Champions* (1.345 khách) đóng góp 68,8% doanh thu ròng. Phân khúc *At Risk* gồm 708 khách hàng từng chi tiêu lớn (£1,56 triệu) nhưng đã ngừng giao dịch trung bình 360,5 ngày — con số này ngụ ý họ là nhóm mua sỉ mùa vụ (Q4/2010) nhưng đã không quay lại vào vụ mùa 2011.

*Khuyến nghị:* Chuẩn hóa toàn bộ hệ thống CRM theo `Net Revenue`. Chủ động gửi catalog Mùa vụ sớm (Tháng 8) cho 708 khách At Risk để kích hoạt lại.

In [41]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pyspark.sql import functions as F

# 1. Tính toán Exact Orders và Net Revenue cấp độ Khách hàng
cust_exact = (sales_enriched
    .filter(F.col('CustomerID').isNotNull())
    .groupBy('CustomerID')
    .agg(
        F.countDistinct('InvoiceNo').alias('Orders'),
        F.sum('Revenue').alias('Gross_Revenue')
    )
)

cust_cancel = (sales_cancelled
    .filter(F.col('CustomerID').isNotNull())
    .groupBy('CustomerID')
    .agg(F.sum('CancelledRevenue').alias('Cancelled_Revenue'))
)

cust_reconciled = (cust_exact
    .join(cust_cancel, on='CustomerID', how='left')
    .na.fill({'Cancelled_Revenue': 0.0})
    .withColumn('Net_Revenue', F.col('Gross_Revenue') - F.col('Cancelled_Revenue'))
    .filter(F.col('Net_Revenue') > 0) # Lọc bỏ các VIP Ảo có Net <= 0
)

# 2. Xây dựng dữ liệu đường cong Pareto
pdf_cust_pareto = cust_reconciled.orderBy(F.desc('Net_Revenue')).toPandas()
total_net_rev = pdf_cust_pareto['Net_Revenue'].sum()

# Tính % lũy kế
pdf_cust_pareto['Cum_Share_100'] = pdf_cust_pareto['Net_Revenue'].cumsum() / total_net_rev * 100
pdf_cust_pareto['Cust_Pct_100'] = np.arange(1, len(pdf_cust_pareto) + 1) / len(pdf_cust_pareto) * 100

# Lấy giá trị Y tại các mốc X quan trọng (1%, 5%, 10%)
idx_1pct = int(len(pdf_cust_pareto) * 0.01) - 1
idx_5pct = int(len(pdf_cust_pareto) * 0.05) - 1
idx_10pct = int(len(pdf_cust_pareto) * 0.10) - 1

val_1pct = pdf_cust_pareto.loc[idx_1pct, 'Cum_Share_100']
val_5pct = pdf_cust_pareto.loc[idx_5pct, 'Cum_Share_100']
val_10pct = pdf_cust_pareto.loc[idx_10pct, 'Cum_Share_100']

# 3. Vẽ biểu đồ Plotly với các mốc Indicators
fig_g3 = go.Figure()

# Vẽ đường cong lũy kế
fig_g3.add_trace(
    go.Scatter(
        x=pdf_cust_pareto['Cust_Pct_100'],
        y=pdf_cust_pareto['Cum_Share_100'],
        name='Tỷ trọng Doanh thu Lũy kế (%)',
        mode='lines',
        line=dict(color='#00CC96', width=3.5),
        fill='tozeroy',
        fillcolor='rgba(0, 204, 150, 0.10)',
        hovertemplate='Top %{x:.2f}% Khách hàng<br>Đóng góp: %{y:.2f}% Doanh thu Ròng<extra></extra>'
    )
)

# Thêm các điểm mốc (Milestones)
milestones = [
    (1.0, val_1pct, "Top 1%"),
    (5.0, val_5pct, "Top 5%"),
    (10.0, val_10pct, "Top 10%"),
    (23.02, 80.0, "Mốc Pareto 80/20")
]

fig_g3.add_trace(
    go.Scatter(
        x=[m[0] for m in milestones],
        y=[m[1] for m in milestones],
        mode='markers+text',
        text=[f"{m[2]}<br>({m[1]:.1f}%)" for m in milestones],
        textposition=['bottom right', 'bottom right', 'bottom right', 'top left'],
        marker=dict(size=10, color='#EF553B', symbol='diamond'),
        showlegend=False,
        hoverinfo='skip'
    )
)

# Thêm đường gióng tham chiếu
fig_g3.add_hline(y=80, line_dash="dash", line_color="#EF553B", annotation_text="Mốc 80% Doanh thu Ròng", annotation_position="top left")
fig_g3.add_vline(x=23.02, line_dash="dot", line_color="#2A3F5F")

fig_g3.update_layout(
    title_text="Hình G.3: Đường cong Tập trung Doanh thu Khách hàng (Đã đối trừ hoàn hủy)",
    height=500,
    template='plotly_white',
    xaxis_title="% Khách hàng Định danh tích lũy (Top %)",
    yaxis_title="% Doanh thu Ròng Lũy kế",
    xaxis=dict(range=[0, 35]),
    yaxis=dict(range=[0, 100]),
    margin=dict(t=60, b=50)
)
fig_g3.show()

## G.4. Phát hiện 4 — Kinh tế học Đơn hàng Xuất khẩu

Thị trường UK đóng góp 85,21% doanh thu nhưng AOV chỉ đạt £489,15/đơn. Ngược lại, các thị trường xuất khẩu thể hiện hai cấu trúc chiến lược:
1.  **AOV siêu cao nhưng độc quyền phân phối:** Hà Lan (AOV £2.430, 22 khách hàng) và EIRE (£1.061, 5 khách hàng định danh). Lô hàng gom lớn tiết kiệm chi phí bưu chính (`POST`) nhưng tạo ra rủi ro đứt gãy nếu mất 1 đại lý.
2.  **Độ phủ rộng, thâm nhập trực tiếp:** Đức (107 khách hàng) và Pháp (95 khách hàng). AOV dao động quanh £540-£570.

*Khuyến nghị:* Ký hợp đồng Master Distributor dài hạn tại Hà Lan/EIRE để khóa rủi ro. Cấp mã ưu đãi vận chuyển theo ngưỡng (£750+) tại Pháp và Đức để kích thích AOV.

In [42]:
country_exact = sales_enriched.groupBy('Country').agg(F.countDistinct('InvoiceNo').alias('Orders'), F.countDistinct('CustomerID').alias('Customers'), F.sum('Revenue').alias('Gross_Revenue'))
country_cancel = sales_cancelled.groupBy('Country').agg(F.sum('CancelledRevenue').alias('Cancelled_Revenue'))

pdf_country = (country_exact
    .join(country_cancel, on='Country', how='left')
    .na.fill({'Cancelled_Revenue': 0.0})
    .withColumn('Net_Revenue', F.col('Gross_Revenue') - F.col('Cancelled_Revenue'))
    .withColumn('AOV', F.col('Gross_Revenue') / F.col('Orders'))
    .filter((F.col('Country') != 'United Kingdom') & (F.col('Orders') >= 30))
    .toPandas()
)

fig_g4 = px.scatter(pdf_country, x="Orders", y="AOV", size="Net_Revenue", text="Country", title="Hình G.4: Ma trận Định vị Thị trường Xuất khẩu (Kích thước = Net Revenue)", size_max=55)
fig_g4.add_hline(y=489.15, line_dash="dash", line_color="#EF553B", annotation_text="AOV Nội địa UK (£489.15)")
fig_g4.update_traces(textposition="top center")
fig_g4.update_layout(height=450, template="plotly_white")
fig_g4.show()

## G.5. Phát hiện 5 — Đặc tính Đuôi nặng (Heavy-Tail) và Ngắt quãng (Sparsity) trong Dự báo

Nhu cầu đặt hàng không phân bổ đều. Việc sử dụng Moving Average hoặc ARIMA sẽ gây ra hiện tượng "làm phẳng" (over-smoothing), dẫn tới thiếu hàng vào ngày nhập sỉ và tồn đọng vào ngày trống đơn.
Mở rộng RFM cho cấp độ Sản phẩm ($SKU_i$):
*   Tỷ lệ ngày trống đơn (Sparsity Rate) $S_i^{\text{lifespan}} = 1 - \frac{D_{\text{active}, i}}{D_{\text{span}, i}}$
*   Ngay cả trong Top 100 SKU bán chạy, Sparsity trung bình đạt 29,04%.

*Khuyến nghị:* Áp dụng Mô hình Dự báo Hai tầng (Croston/Hurdle). Tầng 1 dự báo xác suất phát sinh đơn (Classification), Tầng 2 dự báo quy mô lô đặt sỉ (Regression).

In [43]:
date_bounds = sales_enriched.agg(F.min('Date').alias('Min_Date'), F.max('Date').alias('Max_Date')).first()
sku_sparsity = (sales_enriched
    .filter(F.col('IsMerchandise'))
    .groupBy('StockCode')
    .agg(
        F.countDistinct('InvoiceNo').alias('F_Exact_Orders'),
        F.countDistinct('Date').alias('Active_Days'),
        F.datediff(F.max('Date'), F.min('Date')).alias('Lifespan_Days')
    )
    .withColumn('Sparsity_Lifespan_Pct', F.round((1.0 - F.col('Active_Days') / (F.col('Lifespan_Days') + 1)) * 100, 1))
)

top100_sparsity = sku_sparsity.orderBy(F.desc('F_Exact_Orders')).limit(100).toPandas()
mean_sparsity = top100_sparsity['Sparsity_Lifespan_Pct'].mean()

fig_g5 = px.histogram(top100_sparsity, x='Sparsity_Lifespan_Pct', nbins=18, title="Hình G.5: Phân phối Tỷ lệ Ngày trống đơn (Sparsity) của Top 100 SKU", color_discrete_sequence=['#00CC96'])
fig_g5.add_vline(x=mean_sparsity, line_dash='dash', line_color='#EF553B', annotation_text=f"Trung bình: {mean_sparsity:.1f}%")
fig_g5.update_layout(height=350, template='plotly_white')
fig_g5.show()

## G.6. Tổng hợp Lộ trình Hành động và Hạn chế

Xuyên suốt báo cáo, vấn đề chất lượng dữ liệu được ưu tiên hàng đầu: giao dịch hủy và phân mảnh dữ liệu `Description` có thể làm biến dạng chiến lược. Kế hoạch hành động ưu tiên:

1.  **Khóa dữ liệu Net Revenue:** Loại bỏ rác kế toán (`M`, `DOT`, VIP Ảo) trước khi xuất báo cáo BI.
2.  **Chiến lược Bundling & Phân bổ Tồn kho Quý 4:** Kết hợp Traffic Builders (`21212`) với Cash Cows (`22423`). Áp dụng mô hình dự báo ngắt quãng, nhập hàng an toàn trước 01/09.
3.  **Tái kích hoạt CRM:** Khảo sát nhóm 708 khách At Risk xem đây là churn thực sự hay chỉ là chu kỳ mùa vụ chưa tới.
4.  **Tái cấu trúc Xuất khẩu:** Ký Master Contract cho đại lý lớn (Hà Lan/EIRE), cấp chính sách hỗ trợ phí vận chuyển cho thị trường bán lẻ rộng (Đức/Pháp).

**Hạn chế phương pháp luận (Robustness Limitations):**
Không có `Original_InvoiceNo` nên việc đối khớp giao dịch hủy bị cắt cụt đầu kỳ (left-truncation) ở cấp độ giao dịch đơn lẻ mang sai số, dù tổng thể theo nhóm Customer/SKU được đảm bảo. Thiếu dữ liệu COGS ngăn cản việc phân tích Biên lợi nhuận (Profit Margin) chính xác.
